## Milestone 1: Tackling big data on your laptop

## Overall project goal and data 

During this course, you will be working on an *individual* project involving big data. The purpose is to get exposure to working with much larger datasets than you have previously in MDS. In particular, you will be building and deploying ensemble machine learning models in the cloud to predict daily rainfall in Australia on a large dataset (~6 GB), where features are outputs of different climate models, and the target is the actual rainfall observation.  


You will be using [this dataset on figshare](https://figshare.com/articles/dataset/Daily_rainfall_over_NSW_Australia/14096681). This folder has the output of different climate models as features, and our ultimate goal is to build an ensemble model on these outputs and compare the results with the actual rainfall. At the end of the project, you should have your ML model deployed in the cloud for others to use. 

### Measurement Helper

This notebook utilized a measurement helper. Decorate any function to measure it:

```python
@measure
def combine_csvs(folder):
    ...   # your implementation here

combine_csvs("data/")
```

- **wall** — real elapsed time
- **cpu** — time the processor was actually busy (I/O wait excluded)
- **mem Δ** — change in OS-level Memory (RAM) after the call (RSS)
- we don't have storage I/O estimate here, but when `wall >> cpu`, disk I/O is likely your bottleneck.


In [1]:
# Imports
import time, psutil, os, functools
import re
import glob
import zipfile
import time
from pathlib import Path
import json
import requests
import pandas as pd
from urllib.request import urlretrieve

In [2]:
# Measurement Helper
_proc = psutil.Process(os.getpid())

def measure(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        m0 = _proc.memory_info().rss / 1e6
        c0 = time.process_time()
        t0 = time.perf_counter()
        out = fn(*args, **kwargs)
        print(f"{fn.__name__}:  wall={time.perf_counter()-t0:.1f}s  "
              f"cpu={time.process_time()-c0:.1f}s  "
              f"mem Δ{_proc.memory_info().rss/1e6 - m0:+.0f} MB")
        return out
    return wrapper


### 1. Downloading the data 

1. Download the data from [figshare](https://figshare.com/articles/dataset/Daily_rainfall_over_NSW_Australia/14096681) (look at the URL, and you will see the `article_id` at the end) to your local computer using the [figshare API](https://docs.figshare.com) (you need to make use of `requests` library).

2. Extract the zip file, again programmatically, similar to how we did it in class. 

>  You can download the data and unzip it manually. But we learned about APIs, so we can do it in a reproducible way with the `requests` library, similar to how we [did it in class](https://pages.github.ubc.ca/mds-2025-26/DSCI_525_web-cloud-comp_book/lectures/l1a_rest_api.html#combine).

> There are 5 files in the figshare repo. The one we want is: `data.zip`

In [3]:
# use current notebook directory
WORKDIR = Path.cwd()
OUTPUT_DIR = WORKDIR / "data"
OUTPUT_DIR.mkdir(exist_ok=True)
FORCE_DOWNLOAD = False 

In [4]:
# Necessary metadata
article_id = 14096681
url = f"https://api.figshare.com/v2/articles/{article_id}"
headers = {"Content-Type": "application/json"}

In [5]:
response = requests.request("GET", url, headers=headers)
data = json.loads(response.text)  # this contains all the articles data, feel free to check it out
files = data["files"]             # this is just the data about the files, which is what we want
files

[{'id': 26579150,
  'name': 'daily_rainfall_2014.png',
  'size': 58863,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579150',
  'supplied_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'computed_md5': 'fd32a2ffde300a31f8d63b1825d47e5e',
  'mimetype': 'image/png'},
 {'id': 26579171,
  'name': 'environment.yml',
  'size': 192,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26579171',
  'supplied_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'computed_md5': '060b2020017eed93a1ee7dd8c65b2f34',
  'mimetype': 'text/plain'},
 {'id': 26586554,
  'name': 'README.md',
  'size': 5422,
  'is_link_only': False,
  'download_url': 'https://ndownloader.figshare.com/files/26586554',
  'supplied_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'computed_md5': '61858c6cc0e6a6d6663a7e4c75bbd88c',
  'mimetype': 'text/x-python'},
 {'id': 26766812,
  'name': 'data.zip',
  'size': 814041183,
  'is_link_only': False,
  'download_url': 'https://n

In [6]:
files_to_dl = ["data.zip"] 

for file in files:
    if file["name"] in files_to_dl:
        output_path = OUTPUT_DIR / file["name"]
        
        if output_path.exists() and not FORCE_DOWNLOAD:
            print(f"{file['name']} already exists. Skipping download.")
        else:
            output_path.parent.mkdir(exist_ok=True)  # ensure folder exists

            print(f"Downloading {file['name']}...")
            urlretrieve(file["download_url"], output_path)
            print(f"Downloaded {file['name']} to {output_path}.")

Downloaded data.zip to /Users/jiafuli/Desktop/MDS/Block_6/525/lab/525_ml1/data/data.zip.


In [7]:
zip_path = OUTPUT_DIR / "data.zip"
extract_dir = OUTPUT_DIR / "extracted_data"

if not extract_dir.exists():
    print("Extracting files...")
    extract_dir.mkdir(exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as f:
        f.extractall(extract_dir)
    print("Extraction complete!")
else:
    print("Files already extracted. Skipping extraction.")

Extracting files...
Extraction complete!


In [9]:
%ls -ltr data/extracted_data/

total 10598424
-rw-r--r--   1 jiafuli  staff   95376895  3 24 10:19 MPI-ESM-1-2-HAM_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff   94960113  3 24 10:19 AWI-ESM-1-1-LR_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff   82474546  3 24 10:19 NorESM2-LM_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff  127613760  3 24 10:19 ACCESS-CM2_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff  232118894  3 24 10:19 FGOALS-f3-L_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff  330360682  3 24 10:19 CMCC-CM2-HR4_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff  254009247  3 24 10:19 MRI-ESM2-0_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff  235661418  3 24 10:19 GFDL-CM4_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff  294260911  3 24 10:19 BCC-CSM2-MR_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff  295768615  3 24 10:19 EC-Earth3-Veg-LR_daily_rainfall_NSW.csv
-rw-r--r--   1 jiafuli  staff  328852379  3 24 10:19 CMCC-ESM2_daily_rainfall_NSW.csv
-rw-r--r--  

### 2. Combining data CSVs
1. Combine data CSVs into a single CSV using pandas.
    
2. When combining the CSV files, add an extra column called "model" that identifies the model.
    - Tip 1: you can get this column populated from the file name, eg: for file name "SAM0-UNICON_daily_rainfall_NSW.csv", the model name is SAM0-UNICON
    - Tip 2: Remember how we added "year" column when we combined airline CSVs. Here the regex will be to get word before an underscore ie, `"/([^_]*)"`

> Note: There is a file called `observed_daily_rainfall_SYD.csv` in the data folder that you downloaded. Make sure you exclude this file (programmatically or just take out that file from the folder) before you combine CSVs. We will use this file in our next milestone.

3. Wrap your combine function with `@measure` and ***compare*** wall time, CPU time, and memory across machines. Record results in the table below.

4. Based on the comparison, in 3–5 sentences, reflect on what you observe: how do the numbers differ across machines? What does the `wall`/`cpu` ratio tell you about where the bottleneck is? What surprised you?

| Member | Approach | OS | RAM | Processor | SSD? | wall (s) | cpu (s) | mem Δ (MB) |
|---|---|---|---|---|---|---|---|---|
| 1 (Yusheng Li) | combine |macOS Sequoia 15.7.3 |8 GB|1.4 GHz Quad-core Intel Core i5| Yes |Data B |Data B |Data B |


In [10]:
data_dir = Path("data/extracted_data")

@measure
def combine_csvs(folder_path):
    files = list(folder_path.glob("*.csv"))
    
    combined_df = pd.concat(
        (
            pd.read_csv(file)
            .assign(model=re.findall(r"([^_]+)", file.name)[0])
            for file in files 
            if file.name != "observed_daily_rainfall_SYD.csv"
        ),
        ignore_index=True
    )
    
    output_path = folder_path.parent / "combined_data.csv"
    combined_df.to_csv(output_path, index=False)
    
    print(f"Combined data saved to: {output_path}")
    print(f"Total length: {len(combined_df)}")
    return combined_df

df_combined = combine_csvs(data_dir)

KeyboardInterrupt: 